In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
train = pd.read_csv(r'C:\Users\chill\OneDrive\Documents\ML-DS Competition\StudentHealthRisk\datasets\train.csv')
test = pd.read_csv(r"C:\Users\chill\OneDrive\Documents\ML-DS Competition\StudentHealthRisk\datasets\test.csv")
print("train shape: ", train.shape)
print("test shape: ", test.shape)

train shape:  (690088, 15)
test shape:  (295753, 14)


In [4]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       295753 non-null  int64  
 1   sleep_duration           263182 non-null  float64
 2   heart_rate               292396 non-null  float64
 3   bmi                      289797 non-null  float64
 4   calorie_expenditure      273101 non-null  float64
 5   step_count               289789 non-null  float64
 6   exercise_duration        292795 non-null  float64
 7   water_intake             277120 non-null  float64
 8   diet_type                292795 non-null  object 
 9   stress_level             260263 non-null  object 
 10  sleep_quality            270754 non-null  object 
 11  physical_activity_level  280058 non-null  object 
 12  smoking_alcohol          283504 non-null  object 
 13  gender                   286593 non-null  object 
dtypes: f

In [3]:
train_df = train.iloc[:,1:]
test_df = test.iloc[:,1:]
print("train cols: \n", train_df.columns)
print("\ntest cols:\n", test_df.columns)

train cols: 
 Index(['health_condition', 'sleep_duration', 'heart_rate', 'bmi',
       'calorie_expenditure', 'step_count', 'exercise_duration',
       'water_intake', 'diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol', 'gender'],
      dtype='object')

test cols:
 Index(['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake', 'diet_type',
       'stress_level', 'sleep_quality', 'physical_activity_level',
       'smoking_alcohol', 'gender'],
      dtype='object')


In [4]:
y = train_df["health_condition"]
X = train_df.drop("health_condition", axis=1)

In [5]:
y.unique()

array(['unhealthy', 'at-risk', 'fit'], dtype=object)

In [5]:
y = y.map({
    "fit":0,
    "unhealthy":1,
    "at-risk":2
})
print(y.tail())
print(y.head())

690083    2
690084    2
690085    0
690086    2
690087    2
Name: health_condition, dtype: int64
0    1
1    2
2    1
3    1
4    2
Name: health_condition, dtype: int64


In [17]:
X.columns

Index(['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake', 'diet_type',
       'stress_level', 'sleep_quality', 'physical_activity_level',
       'smoking_alcohol', 'gender'],
      dtype='object')

In [6]:
numerical_cols = []
for cols in X.columns:
    if X[cols].dtypes == "float" or X[cols].dtypes == "int":
        numerical_cols.append(cols)
print("numerical col:", numerical_cols)
print("total columns: ", len(numerical_cols))

numerical col: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
total columns:  7


In [7]:
ordinal_cols = ["stress_level", "sleep_quality", "physical_activity_level"]
categorical_cols = ["diet_type", "smoking_alcohol","gender"]

In [13]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
cat_preprocessor = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

ord_preprocessor = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ("encode", OrdinalEncoder())
])

num_preprocessor = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="median")),
    ("scale", StandardScaler())    
])

preprocessor = ColumnTransformer(transformers=[
    ("cat", cat_preprocessor, categorical_cols),
    ("ord", ord_preprocessor, ordinal_cols),
    ("num", num_preprocessor, numerical_cols)
])

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.22,
    shuffle=True
)
print(f"X train:{X_train.shape}\nX test: {X_test.shape}\ny train: {y_train.shape}\ny test: {y_test.shape}")

X train:(538268, 13)
X test: (151820, 13)
y train: (538268,)
y test: (151820,)


In [18]:
# lightGBMClassifier
from lightgbm import LGBMClassifier
lgbm = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LGBMClassifier(random_state=42, class_weight="balanced", boosting_type="gbdt"))
])

params = {
    'model__learning_rate': [0.01, 0.03, 0.05],
    'model__num_leaves': [21, 31, 41],
    'model__max_depth': [4, 6, -1],
    'model__min_child_samples': [10, 20, 30],
}

lgbm_model = GridSearchCV(
    estimator=lgbm,
    cv=3,
    param_grid=params,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

In [28]:
from sklearn.model_selection import GridSearchCV
dt_pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

params = {
    'model__criterion': ['gini', 'entropy'],
    'model__max_depth': [5, 7, 10],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}

dt_model = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=params,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

In [29]:
dt_model.fit(X_train, y_train)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [5, 7, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...), ('ord', ...), ...]"


In [19]:
lgbm_model.fit(X_train, y_train)

Fitting 3 folds for each of 81 candidates, totalling 243 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__learning_rate': [0.01, 0.03, ...], 'model__max_depth': [4, 6, ...], 'model__min_child_samples': [10, 20, ...], 'model__num_leaves': [21, 31, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...), ('ord', ...), ...]"


In [20]:
best_lgbm = lgbm_model.best_estimator_
print(f"best score: {lgbm_model.best_score_}")
print(f"best parameters: {lgbm_model.best_params_}")

best score: 0.8735016800586278
best parameters: {'model__learning_rate': 0.05, 'model__max_depth': -1, 'model__min_child_samples': 30, 'model__num_leaves': 41}


In [30]:
best_dt = dt_model.best_estimator_
print("best parameters:\n", dt_model.best_params_)
print("best score: \n", dt_model.best_score_)

best parameters:
 {'model__criterion': 'gini', 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2}
best score: 
 0.9541269435546681


In [22]:

def evaluate(model, X_test, y_test):
    from sklearn.metrics import  balanced_accuracy_score
    y_pred = model.predict(X_test)
    accuracy = balanced_accuracy_score(y_test, y_pred)
    return accuracy

In [31]:
evaluate(model=best_dt, X_test=X_test, y_test=y_test)

0.805704882808122

In [23]:
evaluate(model=best_lgbm, X_test=X_test, y_test=y_test)

c:\Users\chill\OneDrive\Documents\ML-DS Competition\comp\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.907357588444749

In [25]:
def output_submission(model, test, name="submission"):
    test_id = test['id']
    test = test.drop('id', axis=1)
    y_pred = model.predict(test)
    file = pd.DataFrame({
        "id": test_id,
        "health_condition": y_pred
    })
    file['health_condition'] = file['health_condition'].map(
        {
            0:'fit',
            1:"unhealthy",
            2:"at-risk"
        })
    file.head(5)
    print('saving file...')
    file.to_csv(f"results\{name}.csv", index=False)
    print(f"{name} file saved")


In [ ]:
output_submission(model=best_dt, test=test, name="submission0")

c:\Users\chill\OneDrive\Documents\ML-DS Competition\comp\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


saving file...
submission1 file saved


In [27]:
# save lgbm model
import joblib
joblib.dump(value=best_lgbm, filename=r"results/lgbm.joblib")
print("model saved...")

model saved...
